# E4 — Ablations and E5 — thread scaling

**E4** (`IMPLEMENTATION_PLAN.md` §14.1): which parts of LA-ef matter? Each axis is varied one at a time around the `laef_wf` baseline at ρ ∈ {1.2, 2.0}: controller {p_aimd, fixed_aimd, static_capacity}, queue guard on/off, floor mode on/off, allocator {water_filling, uniform_cut}, window n, growth step, ladder ratio.

**E5:** hardware contention — unit cost u(C) and capacity B_max(C) = C / u(C) as the worker count grows.

**Inputs:**
* `results/e4_ablations/summary.csv` — run parameters (`dataset, policy, threads, slo_mult, profile, rho, repeat, axis, axis_value`; the baseline rows have `axis = base`) + every `summary.json` key;
* `results/e5_threads/summary.csv` — same layout, `axis = threads`;
* `data/artifacts/<dataset>/profile/profile_C{C}.json` — `{threads, idle: {...}, saturation: {mu_sat_qps, u_s_per_dc, mean_service_s, s_p95_s}}`.

In [ ]:
import json
import pathlib
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
ARTIFACTS = ROOT / 'data' / 'artifacts'
FIG = ROOT / 'results' / 'figures' / 'e4_e5'
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

def read_summary(exp):
    p = RESULTS / exp / 'summary.csv'
    if not p.exists():
        print(f'missing {p}: run  python scripts/run_experiments.py configs/experiments/e4_ablations.yaml')
        return pd.DataFrame()
    return pd.read_csv(p)

e4 = read_summary('e4_ablations')
e5 = read_summary('e5_threads')
print('E4 runs:', len(e4), '| E5 runs:', len(e5))
if len(e4):
    print(e4.groupby('axis').axis_value.unique())

## 1. E4 — one panel row per ablation axis

Bars are means over repeats (± std). The baseline (`laef_wf` defaults) is the value marked with ★. Every query is served in every variant, so the trade-off is purely recall vs latency.

In [ ]:
METRICS = [('mean_recall', 'mean recall'), ('p5_recall', 'P5 recall'), ('slo_attainment', 'SLO attainment'),
           ('p95_ms', 'p95 latency [ms]'), ('recovery_time_s', 'recovery [s]')]
BASELINE = {'controller.type': 'p_aimd', 'controller.guard': 'True', 'floor_mode.enabled': 'True',
            'allocator.type': 'water_filling', 'window.n': '500', 'controller.growth': '1.02',
            'calibration.ladder_ratio': '2.0'}

def with_baseline(df, axis):
    # rows varying `axis`, plus the shared baseline rows relabelled with the baseline value
    rows = df[df.axis == axis].copy()
    base = df[df.axis == 'base'].copy()
    if len(base):
        base['axis_value'] = BASELINE.get(axis, 'base')
        rows = pd.concat([rows, base])
    rows['axis_value'] = rows.axis_value.astype(str)
    return rows

if len(e4):
    axes_list = [a for a in e4.axis.unique() if a != 'base']
    for (ds, rho), d in e4.groupby(['dataset', 'rho']):
        fig, ax = plt.subplots(len(axes_list), len(METRICS), figsize=(3.4 * len(METRICS), 2.6 * len(axes_list)), squeeze=False)
        for i, axis in enumerate(axes_list):
            rows = with_baseline(d, axis)
            for j, (m, label) in enumerate(METRICS):
                a = ax[i, j]
                if m not in rows or rows[m].isna().all():
                    a.set_visible(False)
                    continue
                g = rows.groupby('axis_value')[m].agg(['mean', 'std'])
                colors = ['tab:green' if v == BASELINE.get(axis) else 'tab:gray' for v in g.index]
                a.bar(range(len(g)), g['mean'], yerr=g['std'].fillna(0), color=colors, capsize=2)
                a.set_xticks(range(len(g)), [('★ ' if v == BASELINE.get(axis) else '') + v for v in g.index], fontsize=7, rotation=20)
                if i == 0:
                    a.set_title(label)
                if j == 0:
                    a.set_ylabel(axis, fontsize=8)
        fig.suptitle(f'E4 ablations — {ds}, ρ = {rho:g}', y=1.0)
        fig.tight_layout()
        fig.savefig(FIG / f'e4_{ds}_rho{rho:g}.png', bbox_inches='tight')
        plt.show()

## 2. E4 — effect sizes relative to the baseline

Difference (variant − baseline) for each axis value, averaged over repeats. Positive Δ recall / Δ attainment is better; positive Δ p95 is worse.

In [ ]:
if len(e4):
    keys = ['mean_recall', 'p5_recall', 'p1_recall', 'slo_attainment', 'p95_ms', 'guard_fires', 'floor_mode_windows']
    keys = [k for k in keys if k in e4]
    out = []
    for (ds, rho), d in e4.groupby(['dataset', 'rho']):
        base = d[d.axis == 'base'][keys].mean()
        for (axis, val), s in d[d.axis != 'base'].groupby(['axis', 'axis_value']):
            out.append({'dataset': ds, 'rho': rho, 'axis': axis, 'value': val,
                        **{f'Δ {k}': s[k].mean() - base[k] for k in keys}})
    effects = pd.DataFrame(out)
    effects.to_csv(FIG / 'e4_effects.csv', index=False)
    display(effects.round(4))

## 3. E4 — controller comparison over time

B(t) and window p95 for the three budget methods (first repeat), from each run's `timeseries.csv`. Run directories are located through the `run_id` column of `summary.csv`.

In [ ]:
if len(e4) and 'run_id' in e4:
    ctl = e4[(e4.axis == 'controller.type') | (e4.axis == 'base')]
    for (ds, rho), d in ctl.groupby(['dataset', 'rho']):
        fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
        for _, row in d[d.repeat == d.repeat.min()].iterrows():
            name = BASELINE['controller.type'] if row.axis == 'base' else row.axis_value
            ts_path = RESULTS / 'e4_ablations' / str(row.run_id) / 'timeseries.csv'
            if not ts_path.exists():
                continue
            ts = pd.read_csv(ts_path)
            ax[0].plot(ts.t_s, ts.B / 1e6, label=name)
            ax[1].plot(ts.t_s, ts.p95_ms, label=name)
        if 'slo_ms' in d:
            ax[1].axhline(d.slo_ms.iloc[0], color='k', ls='--', lw=1)
        ax[0].set(xlabel='time [s]', ylabel='B [M dc/s]', title=f'{ds}, ρ={rho:g}: budget')
        ax[1].set(xlabel='time [s]', ylabel='window p95 [ms]', yscale='log', title='latency')
        ax[0].legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(FIG / f'e4_{ds}_rho{rho:g}_controllers.png')
        plt.show()

## 4. E5 — unit cost and capacity vs threads

From the saturation profiles: if distance computations did not contend for memory bandwidth / cache, u(C) would stay flat and B_max(C) would grow linearly in C (dotted line).

In [ ]:
prof_rows = []
for ds_dir in sorted(ARTIFACTS.glob('*')) if ARTIFACTS.exists() else []:
    for p in sorted((ds_dir / 'profile').glob('profile_C*.json')):
        j = json.loads(p.read_text())
        sat, idle = j.get('saturation', {}), j.get('idle', {})
        C = j.get('threads') or int(re.search(r'C(\d+)', p.stem).group(1))
        u = sat.get('u_s_per_dc', np.nan)
        prof_rows.append({'dataset': ds_dir.name, 'C': C, 'u_ns': 1e9 * u, 'B_max_Mdc_s': C / u / 1e6 if u else np.nan,
                          'mu_sat_qps': sat.get('mu_sat_qps', np.nan), 'u_idle_ns': 1e9 * idle.get('u_s_per_dc', np.nan),
                          'S_p95_sat_ms': 1e3 * sat.get('s_p95_s', np.nan)})
prof = pd.DataFrame(prof_rows)
if len(prof):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    for ds, d in prof.sort_values('C').groupby('dataset'):
        l, = ax[0].plot(d.C, d.u_ns, '-o', ms=3, label=ds)
        ax[1].plot(d.C, d.B_max_Mdc_s, '-o', ms=3, color=l.get_color())
        c1 = d.iloc[0]
        ax[1].plot(d.C, c1.B_max_Mdc_s * d.C / c1.C, ':', color=l.get_color())
        ax[2].plot(d.C, d.mu_sat_qps, '-o', ms=3, color=l.get_color())
    ax[0].set(xlabel='threads C', ylabel='u [ns / distance]', title='unit cost u(C)')
    ax[1].set(xlabel='threads C', ylabel='B_max [M dc/s]', title='capacity C/u (dotted: linear)')
    ax[2].set(xlabel='threads C', ylabel='μ_sat [QPS]', title='saturation throughput (Ada-ef)')
    ax[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG / 'e5_profiles.png')
    plt.show()
    display(prof.sort_values(['dataset', 'C']).round(3))
else:
    print('no profiles found under', ARTIFACTS)

## 5. E5 — LA-ef under load as threads grow (ρ = 1.2)

In [ ]:
if len(e5):
    keys = [k for k in ['mean_recall', 'p5_recall', 'slo_attainment', 'p95_ms', 'achieved_qps'] if k in e5]
    g = e5.groupby(['dataset', 'threads'])[keys].mean().reset_index()
    fig, ax = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 3.4))
    for ds, d in g.groupby('dataset'):
        for a, k in zip(ax, keys):
            a.plot(d.threads, d[k], '-o', ms=3, label=ds)
            a.set(xlabel='threads C', title=k)
    ax[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIG / 'e5_laef.png')
    plt.show()
    display(g.round(3))